# 04 — Baseline: Logistic Regression Credit Scorecard
**Owner:** Member 3  |  **Phase:** 3  |  **Date:** May 12

Objectives: reproduce starter baseline, extend with WoE, run 5-fold CV, generate submission.

In [ ]:
import sys, pathlib
import numpy as np
import pandas as pd
sys.path.insert(0, str(pathlib.Path(".").resolve()))

from src.data_loader import load_train, load_test
from src.preprocessing import load_processed
from src.woe_iv import iv_summary, encode_woe
from src.models import get_logreg
from src.train import load_folds, cross_validate_model
from src.evaluate import compute_metrics
from src.submit import make_submission, validate_submission

train_proc, test_proc = load_processed()

## 1. Reproduce Starter Baseline (7 numeric features only)

In [ ]:
from sklearn.model_selection import train_test_split
NUM_FEATS = ["amount_usd","annual_rate_pct","term_months","monthly_income_usd",
              "existing_obligations","num_dependents","months_at_employer"]
X = train_proc[NUM_FEATS].fillna(train_proc[NUM_FEATS].median())
y = train_proc["Target"].values

X_tr, X_val, y_tr, y_val = train_test_split(X, y, test_size=0.3, random_state=42, stratify=y)
logreg = get_logreg()
logreg.fit(X_tr, y_tr)
val_proba = logreg.predict_proba(X_val)[:,1]
print("Baseline (7 features):", compute_metrics(y_val, val_proba))

## 2. WoE Scorecard — All Features with IV >= 0.02

In [ ]:
from src.feature_engineering import engineer_all_features
train_raw = load_train()
test_raw  = load_test()
train_raw = engineer_all_features(train_raw)
test_raw  = engineer_all_features(test_raw)

all_feats = [c for c in train_raw.columns
             if c not in ["Target","ID"] and str(train_raw[c].dtype) != "datetime64[ns]"]
cat_feats = list(train_raw[all_feats].select_dtypes(include="object").columns)
iv_df     = iv_summary(train_raw, all_feats, "Target", cat_features=cat_feats)
keep_feats = iv_df[iv_df["iv"] >= 0.02]["feature"].tolist()
print(f"Keeping {len(keep_feats)} features")

In [ ]:
train_woe, test_woe = encode_woe(train_raw, test_raw, keep_feats, "Target", cat_features=cat_feats)
woe_cols = [c + "_woe" for c in keep_feats]
X_woe = train_woe[woe_cols].values
y     = train_raw["Target"].values

folds   = load_folds()
results = cross_validate_model(get_logreg(), X_woe, y, folds)
print(f"\nWoE Scorecard CV AUC: {results['mean_auc']:.5f} +/- {results['std_auc']:.5f}")

## 3. Generate Submission

In [ ]:
logreg_full = get_logreg()
logreg_full.fit(X_woe, y)
X_test_woe = test_woe[woe_cols].values
test_preds = logreg_full.predict_proba(X_test_woe)[:,1]
sub = make_submission(test_raw["ID"], test_preds, label="logreg_woe")
print(sub.head())